# Silver Layer
# Läser från bronze, rensar data och skapar OBT med IDs


In [0]:

BRONZE_TABLE = "marathos.bronze.ultra_marathon_raw"
SILVER_TABLE  = "marathos.silver.ultra_marathon_obt"


df = spark.table(BRONZE_TABLE)
print(f"Rader från bronze: {df.count():,}")
display(df.limit(3))

In [0]:
from pyspark.sql.functions import col, regexp_extract, when


df.groupBy("event_distance_length").count().orderBy("count", ascending=False).limit(20).display()

In [0]:
from pyspark.sql.functions import col, when, dense_rank, regexp_extract, to_timestamp
from pyspark.sql.window import Window


df_clean = df.withColumn(
    "distance_unit",
    when(col("event_distance_length").endswith("km"), "km")
    .when(col("event_distance_length").endswith("mi"), "mi")
    .when(col("event_distance_length").endswith("h"), "h")
    .otherwise("unknown")
)


df_clean = df_clean.withColumn(
    "is_valid",
    when(
        (col("distance_unit").isin("km", "mi")) & 
        (col("athlete_performance").endswith("h")), True
    ).when(
        (col("distance_unit") == "h") & 
        (~col("athlete_performance").endswith("h")), True
    ).otherwise(False)
)


df_clean = df_clean.filter(col("is_valid") == True)

print(f"Rader efter rensning: {df_clean.count():,}")

In [0]:

from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window


w_event = Window.orderBy("event_name")
df_clean = df_clean.withColumn("event_id", dense_rank().over(w_event))


w_athlete = Window.orderBy("athlete_id")
df_clean = df_clean.withColumn("athlete_id_new", dense_rank().over(w_athlete))


from pyspark.sql.functions import monotonically_increasing_id
df_clean = df_clean.withColumn("result_id", monotonically_increasing_id())

print("IDs skapade!")
df_clean.select("result_id", "event_id", "event_name", "athlete_id_new").limit(5).display()

In [0]:

(df_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE))

print(f"Silver-tabell sparad: {SILVER_TABLE}")
spark.table(SILVER_TABLE).count()

In [0]:
from pyspark.sql.functions import dense_rank, regexp_extract, col, when
from pyspark.sql.window import Window

df_typed = df_clean.withColumn(
    "performance_value",
    when(
        col("distance_unit").isin("km", "mi"),
        regexp_extract(col("athlete_performance"), r"(\d+:\d+:\d+)", 1)
    ).otherwise(
        regexp_extract(col("athlete_performance"), r"([\d\.]+)", 1)
    )
)


w_result = Window.orderBy("event_id", "athlete_id_new", "athlete_performance")
df_typed = df_typed.withColumn("result_id", dense_rank().over(w_result))

print("Datatyper konverterade!")
df_typed.select("result_id", "event_id", "athlete_performance", "performance_value", "distance_unit").limit(5).display()

In [0]:
(df_typed.write
         .format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable(SILVER_TABLE))

print(f"Silver-tabell uppdaterad: {SILVER_TABLE}")
print(f"Antal rader: {spark.table(SILVER_TABLE).count():,}")

In [0]:
%sql
USE CATALOG marathos;
SHOW VIEWS IN gold;

In [0]:
%sql
SELECT * FROM marathos.silver.ultra_marathon_obt LIMIT 5

In [0]:
%sql
SELECT column_name 
FROM marathos.information_schema.columns 
WHERE table_name = 'ultra_marathon_obt'
ORDER BY ordinal_position

In [0]:
%sql
SELECT is_valid, COUNT(*) 
FROM marathos.silver.ultra_marathon_obt 
GROUP BY is_valid